# Task 1 - Financial AI Equity Research Assistant

**Ticker:** AAPL

Modules: `data_pipeline.py`, `llm_reasoning.py`, `prompts.py`, `schemas.py`, `report.py`.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / 'task1_financial').exists():
    sys.path.insert(0, str(ROOT / 'task1_financial'))
else:
    sys.path.insert(0, str(ROOT))

from data_pipeline import run_pipeline, SMA_FAST, SMA_SLOW, RSI_PERIOD
pipe = run_pipeline('AAPL', period='2y')
print('OHLCV rows:', len(pipe['ohlcv']))
print('Featured columns:', list(pipe['featured'].columns))
print('Headlines:', len(pipe['headlines']))
print('Summary:')
pipe['summary']

OHLCV rows: 501
Featured columns: ['open', 'high', 'low', 'close', 'volume', 'sma_50', 'sma_200', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_mid', 'bb_upper', 'bb_lower']
Headlines: 10
Summary:


{'ticker': 'AAPL',
 'as_of': '2026-09-11 00:00:00-04:00',
 'current_price': 332.2699890136719,
 'fifty_two_week_high': 344.57000732421875,
 'fifty_two_week_low': 226.64999389648438,
 'pe_ratio': 38.148106,
 'ytd_return': 0.2260432347167758,
 'momentum_signal': 'bullish',
 'indicators': {'sma_50': 317.97419982910156,
  'sma_200': 284.9533494567871,
  'rsi_14': 62.789009402560566,
  'macd': 3.226967132801235,
  'macd_signal': 1.9022506249270499,
  'macd_hist': 1.3247165078741852,
  'bb_mid': 316.62599792480466,
  'bb_upper': 331.4729512408738,
  'bb_lower': 301.77904460873555}}

## Task 1B - LLM sentiment and signal (Pydantic validated)

In [2]:
from llm_reasoning import run_llm_reasoning, has_groq_key
print('LLM mode:', 'groq' if has_groq_key() else 'offline_fallback')
llm = run_llm_reasoning('AAPL', pipe['summary'], pipe['headlines'])
print('Overall sentiment:', llm['sentiment']['overall_label'], llm['sentiment']['overall_sentiment_score'])
print('Signal:', llm['signal'])
llm['sentiment']['items'][:3]

LLM mode: groq
Overall sentiment: positive 0.25153374233128833
Signal: {'signal': 'Hold', 'justification': 'Apple’s price sits above both the 50‑ and 200‑day SMAs, confirming a bullish trend, and the MACD histogram is positive, indicating upward momentum. The RSI is in the 60s, suggesting moderate overbought conditions but not extreme, while the price is just above the upper Bollinger Band, hinting at a short‑term mean‑reversion risk. News sentiment is mildly positive, providing some support but not enough to override the technical caution. Therefore, a Hold recommendation is prudent until the price consolidates or pulls back toward the Bollinger middle band.', 'key_factors': ['Price above SMA50 and SMA200 (bullish trend)', 'Positive MACD histogram (momentum)', 'RSI in 60s (moderate overbought)', 'Price above upper Bollinger Band (mean‑reversion risk)', 'Mildly positive news sentiment']}


[{'headline': "Apple's AI vision is tailored for the moment of AI panic",
  'sentiment': 'negative',
  'confidence': 0.8,
  'brief_reason': 'Headline references AI panic, indicating negative tone'},
 {'headline': 'Louis Navellier is buying 3 headline-making stocks, including Google',
  'sentiment': 'positive',
  'confidence': 0.8,
  'brief_reason': 'Buying headline-making stocks suggests optimism.'},
 {'headline': 'MP Materials Nears NdPr Target, Eyes GM Magnet Deliveries by Year-End',
  'sentiment': 'positive',
  'confidence': 0.9,
  'brief_reason': 'Company is close to its target and plans deliveries, indicating growth.'}]

## Bonus - Research brief with chart

In [3]:
from report import render_html_report
paths = render_html_report(
    'AAPL', pipe['summary'], llm['sentiment'], llm['signal'],
    pipe['headlines'], pipe['featured'], output_dir='reports'
)
print(paths)

{'markdown': WindowsPath('reports/AAPL_research_brief.md'), 'html': WindowsPath('reports/AAPL_research_brief.html'), 'chart': WindowsPath('reports/AAPL_price_chart.png')}
